# Empezamos con el proceso de reducción dimensional

In [1]:

from pathlib import Path
from datetime import datetime

RUTA_PROYECTO = Path(
    r"C:\Users\marco\Documentos\investigacion"
    r"\machine_learning_idalina\6_redes_neuronales"
)

RUTA_DATOS_RAW = RUTA_PROYECTO / "2_datos" / "1_raw"

RUTA_PROCESADOS = RUTA_PROYECTO / "2_datos" / "2_procesados"

RUTA_RESULTADOS = RUTA_PROYECTO / "3_resultados"

RUTA_DATOS_RAW.mkdir(parents=True, exist_ok=True)
RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Proyecto:")
print(RUTA_PROYECTO)

print("\nDatos originales:")
print(RUTA_DATOS_RAW)

print("\nDatos procesados:")
print(RUTA_PROCESADOS)

print("\nResultados MLP:")
print(RUTA_RESULTADOS)


Proyecto:
C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales

Datos originales:
C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\1_raw

Datos procesados:
C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados

Resultados MLP:
C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\3_resultados


In [4]:
import pandas as pd  

UBICACION_DATOS = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\1_red_neuronal_1_sklearn_mlp\2_datos\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"

df = pd.read_excel(UBICACION_DATOS,   parse_dates=["fecha"])

df = df.sort_values("fecha").reset_index(drop=True)

print("Filas:", len(df))
print("Columnas:", len(df.columns))

df.head()

Filas: 270
Columnas: 172


,fecha,año,semana_epi,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,...,casos_dengue_lag_3,casos_dengue_lag_4,casos_dengue_lag_5,casos_dengue_lag_6,casos_dengue_lag_7,casos_dengue_lag_8,casos_dengue_lag_9,casos_dengue_lag_10,casos_dengue_lag_11,casos_dengue_lag_12
0,2021-03-28,2021,13,31.554286,36.747143,26.918571,15.725714,55.717143,0.67,0,...,0,1,0,1,1,0,0,1,0,0
1,2021-04-04,2021,14,29.200000,33.907143,25.458571,16.550000,66.907143,5.02,2,...,0,0,1,0,1,1,0,0,1,0
2,2021-04-11,2021,15,28.248571,32.278571,25.132857,17.790000,75.144286,40.07,5,...,0,0,0,1,0,1,1,0,0,1
3,2021-04-18,2021,16,29.072857,34.704286,25.035714,17.598571,72.110000,22.25,1,...,1,0,0,0,1,0,1,1,0,0
4,2021-04-25,2021,17,29.080000,34.357143,25.041429,16.558571,67.494286,1.46,1,...,0,1,0,0,0,1,0,1,1,0


In [5]:
df.columns.to_numpy 

<bound method IndexOpsMixin.to_numpy of Index(['fecha', 'año', 'semana_epi', 'temp', 'temp_max', 'temp_min', 'hum_esp',
       'hum_rel', 'prec', 'dias_lluvia',
       ...
       'casos_dengue_lag_3', 'casos_dengue_lag_4', 'casos_dengue_lag_5',
       'casos_dengue_lag_6', 'casos_dengue_lag_7', 'casos_dengue_lag_8',
       'casos_dengue_lag_9', 'casos_dengue_lag_10', 'casos_dengue_lag_11',
       'casos_dengue_lag_12'],
      dtype='str', length=172)>

In [6]:
df_columns = pd.DataFrame(df.columns.to_numpy(), columns=['columnas'])
df_columns 

,columnas
0,fecha
1,año
2,semana_epi
3,temp
4,temp_max
...,...
167,casos_dengue_lag_8
168,casos_dengue_lag_9
169,casos_dengue_lag_10
170,casos_dengue_lag_11


In [7]:
df_columns.to_excel(r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\1_raw\columnas_dataset_rezago.xlsx", index = False)

# Diferentes formas de reducción dimensional para una MLP

In [9]:

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import os
from datetime import datetime


# ============================================================
# 0. CONFIGURACIÓN DE RUTAS
# ============================================================
RUTA_EXCEL = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\1_red_neuronal_1_sklearn_mlp\2_datos\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"
RUTA_PROCESADOS = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados"

# Crear carpeta si no existe
os.makedirs(RUTA_PROCESADOS, exist_ok=True)

# Variables climáticas base (ya vienen con sus lags 1-12 en el Excel de entrada)
VARIABLES_BASE = [
    "temp", "temp_max", "temp_min", "hum_esp", "hum_rel",
    "prec", "dias_lluvia", "vel_vi", "vel_vi_max", "vel_vi_min",
    "soi", "sst",
]

# Columnas que no deben tratarse como features candidatas para la reducción
COLUMNAS_NO_FEATURE = ["fecha", "casos_dengue"]


# ============================================================
# 1. FUNCIONES DE CARGA Y PREPROCESAMIENTO
# ============================================================
def cargar_datos():
    df = pd.read_excel(RUTA_EXCEL, parse_dates=["fecha"])
    print(f"Datos reales cargados ({len(df)} filas, {df.shape[1]} columnas).")
    return df.sort_values("fecha").reset_index(drop=True)


def split_temporal(df, proporcion_train=0.8):
    n = len(df)
    corte = int(n * proporcion_train)
    return df.iloc[:corte].copy(), df.iloc[corte:].copy()


def limpiar_nan(df, method='ffill'):
    """
    Limpia valores NaN en el dataset
    method: 'ffill' (forward fill), 'bfill' (backward fill),
            'drop', 'zero', 'mean'
    """
    df_clean = df.copy()

    if method == 'ffill':
        df_clean = df_clean.ffill()
        df_clean = df_clean.fillna(0)
    elif method == 'bfill':
        df_clean = df_clean.bfill()
        df_clean = df_clean.fillna(0)
    elif method == 'drop':
        df_clean = df_clean.dropna()
    elif method == 'zero':
        df_clean = df_clean.fillna(0)
    elif method == 'mean':
        for col in df_clean.columns:
            if df_clean[col].dtype in ['float64', 'int64']:
                mean_val = df_clean[col].mean()
                df_clean[col] = df_clean[col].fillna(mean_val)

    df_clean = df_clean.fillna(0)
    return df_clean


def crear_features_avanzadas(df):
    """
    Crea features derivadas adicionales A PARTIR del dataset de 172 columnas
    ya provisto (que incluye fecha, año, semana_epi, las 12 variables
    climáticas base, casos_dengue y sus lags 1-12).

    IMPORTANTE: aquí NO se vuelven a calcular los lags de las variables
    base ni de casos_dengue, porque ya existen en el archivo de entrada.
    Solo se agregan agregaciones móviles, ratios, interacciones y
    estacionalidad.
    """
    df_adv = df.copy()

    # 1. Agregaciones móviles de casos (ventanas de 2, 4, 8 semanas)
    for window in [2, 4, 8]:
        df_adv[f'casos_media_{window}w'] = df_adv['casos_dengue'].rolling(window).mean()
        df_adv[f'casos_max_{window}w'] = df_adv['casos_dengue'].rolling(window).max()
        df_adv[f'casos_tendencia_{window}w'] = df_adv['casos_dengue'].diff(window)

    # 2. Features climáticas agregadas (ventanas de 4, 8, 12 semanas)
    for var in ['temp', 'prec', 'hum_rel']:
        for window in [4, 8, 12]:
            df_adv[f'{var}_media_{window}w'] = df_adv[var].rolling(window).mean()
            df_adv[f'{var}_max_{window}w'] = df_adv[var].rolling(window).max()

    # 3. Razones y ratios climáticos
    df_adv['prec_temp_ratio'] = df_adv['prec'] / (df_adv['temp'] + 0.1)
    df_adv['hum_temp_interaction'] = df_adv['hum_rel'] * df_adv['temp']
    df_adv['temp_range'] = df_adv['temp_max'] - df_adv['temp_min']

    # 4. Indicadores de estacionalidad adicionales
    #    (año y semana_epi ya vienen en el dataset original; se agrega
    #    además el mes calendario y la semana ISO del año)
    df_adv['mes'] = df_adv['fecha'].dt.month
    df_adv['semana_del_ano'] = df_adv['fecha'].dt.isocalendar().week.astype(int)

    return df_adv


# ============================================================
# 2. FUNCIONES DE REDUCCIÓN DIMENSIONAL
# ============================================================
def seleccionar_por_importancia(df, features, threshold=0.01):
    """
    Selecciona features con importancia > threshold usando Random Forest
    """
    X = df[features].values
    y = df['casos_dengue'].values

    imp = SimpleImputer(strategy='median')
    X_clean = imp.fit_transform(X)

    rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    rf.fit(X_clean, y)

    importancias = pd.DataFrame({
        'feature': features,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)

    selected = importancias[importancias['importance'] > threshold]['feature'].tolist()

    print(f"  Importancia > {threshold}: seleccionadas {len(selected)} features")
    if len(selected) > 0:
        print(f"  Top 5: {selected[:5]}")
    return selected, importancias


def seleccionar_rfe(df, features, n_features=15):
    """
    Selección recursiva de features con Random Forest
    """
    X = df[features].values
    y = df['casos_dengue'].values

    imp = SimpleImputer(strategy='median')
    X_clean = imp.fit_transform(X)

    rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    selector = RFE(rf, n_features_to_select=n_features, step=1)
    selector.fit(X_clean, y)

    selected = [features[i] for i in range(len(features)) if selector.support_[i]]
    print(f"  RFE seleccionó {len(selected)} features")
    return selected


def seleccionar_lags_optimos(df):
    """
    Encuentra los lags más importantes de casos_dengue (ya presentes en el
    dataset de entrada como casos_dengue_lag_1 ... casos_dengue_lag_12)
    """
    lags = [f'casos_dengue_lag_{lag}' for lag in range(1, 13)
            if f'casos_dengue_lag_{lag}' in df.columns]

    if len(lags) == 0:
        print("  No se encontraron lags de casos_dengue en el dataset de entrada")
        return []

    X = df[lags].values
    y = df['casos_dengue'].values

    imp = SimpleImputer(strategy='median')
    X_clean = imp.fit_transform(X)

    correlaciones = []
    for i, col in enumerate(lags):
        corr = np.corrcoef(X_clean[:, i], y)[0, 1]
        if not np.isnan(corr):
            correlaciones.append((col, abs(corr)))

    if len(correlaciones) == 0:
        return []

    correlaciones.sort(key=lambda x: x[1], reverse=True)
    selected = [col for col, corr in correlaciones if corr > 0.3]
    print(f"  Lags seleccionados: {len(selected)}")
    return selected


def aplicar_pca(df, features, n_components=10):
    """
    Aplica PCA para reducir dimensionalidad
    Retorna: df_pca, pca_cols, pca_model, explained_variance
    """
    X = df[features].values

    imp = SimpleImputer(strategy='median')
    X_clean = imp.fit_transform(X)

    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_clean)

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    explained_variance = pca.explained_variance_ratio_
    print(f"  PCA: {n_components} componentes explican {explained_variance.sum():.2%}")

    pca_cols = [f'PC_{i+1}' for i in range(n_components)]
    df_pca = pd.DataFrame(X_pca, columns=pca_cols)
    df_pca['fecha'] = df['fecha'].values
    df_pca['casos_dengue'] = df['casos_dengue'].values

    return df_pca, pca_cols, pca, explained_variance


# ============================================================
# 3. FUNCIÓN PARA GUARDAR DATASETS REDUCIDOS
# ============================================================
def guardar_dataset_reducido(df, features_seleccionadas, metodo, total_features_originales, descripcion=""):
    """
    Guarda el dataset reducido y la lista de atributos seleccionados
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    columnas_guardar = ['fecha', 'casos_dengue'] + features_seleccionadas
    df_reducido = df[columnas_guardar].copy()

    nombre_base = f"dengue_reducido_{metodo}"
    if descripcion:
        nombre_base += f"_{descripcion}"
    nombre_base += f"_{timestamp}"

    archivo_excel = os.path.join(RUTA_PROCESADOS, f"{nombre_base}.xlsx")
    df_reducido.to_excel(archivo_excel, index=False)
    print(f"  ✅ Dataset reducido guardado: {archivo_excel}")

    archivo_atributos = os.path.join(RUTA_PROCESADOS, f"{nombre_base}_atributos.xlsx")
    df_atributos = pd.DataFrame({
        'feature': features_seleccionadas,
        'orden': range(1, len(features_seleccionadas) + 1)
    })
    df_atributos.to_excel(archivo_atributos, index=False)
    print(f"  ✅ Lista de atributos guardada: {archivo_atributos}")

    # Reducción calculada contra el total de features CANDIDATAS antes de
    # reducir (antes este cálculo comparaba n contra n y siempre daba 0%)
    reduccion_pct = (1 - len(features_seleccionadas) / total_features_originales) * 100

    archivo_resumen = os.path.join(RUTA_PROCESADOS, f"{nombre_base}_resumen.xlsx")
    df_resumen = pd.DataFrame({
        'metadato': ['Método', 'Fecha', 'Total features candidatas', 'Features seleccionadas',
                     'Reducción (%)', 'Descripción'],
        'valor': [metodo, timestamp, total_features_originales,
                  len(features_seleccionadas),
                  f"{reduccion_pct:.1f}%",
                  descripcion]
    })
    df_resumen.to_excel(archivo_resumen, index=False)
    print(f" Resumen guardado: {archivo_resumen}")

    return archivo_excel, archivo_atributos, archivo_resumen


# ============================================================
# 4. FUNCIÓN PRINCIPAL DE REDUCCIÓN DIMENSIONAL
# ============================================================
def reducir_dimensionalidad_y_guardar(df_completo, metodo='importancia', **kwargs):
    """
    Aplica reducción dimensional y guarda los resultados

    Parámetros:
    - metodo: 'importancia', 'rfe', 'lags', 'pca', 'combinado'
    - kwargs: parámetros específicos del método
    """
    print(f"\n{'='*70}")
    print(f"REDUCCIÓN DIMENSIONAL - MÉTODO: {metodo.upper()}")
    print(f"{'='*70}")

    # Crear features avanzadas SOBRE el dataset de 172 columnas ya provisto
    print("\n1️⃣ Creando features avanzadas (agregaciones, ratios, estacionalidad)...")
    df_adv = crear_features_avanzadas(df_completo)
    df_adv = limpiar_nan(df_adv, method='ffill')

    assert df_adv.isna().sum().sum() == 0, "Aún hay NaN en los datos!"

    cols = [col for col in df_adv.columns if col not in COLUMNAS_NO_FEATURE]
    print(f"  Features candidatas totales (172 originales + derivadas): {len(cols)}")

    features_seleccionadas = []
    descripcion = ""

    if metodo == 'importancia':
        threshold = kwargs.get('threshold', 0.005)
        features_seleccionadas, importancias = seleccionar_por_importancia(
            df_adv, cols, threshold=threshold
        )
        descripcion = f"threshold_{threshold}"

        archivo_importancias = os.path.join(RUTA_PROCESADOS,
            f"importancias_features_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        importancias.to_excel(archivo_importancias, index=False)
        print(f"  ✅ Importancias guardadas: {archivo_importancias}")

    elif metodo == 'rfe':
        n_features = kwargs.get('n_features', 20)
        features_seleccionadas = seleccionar_rfe(df_adv, cols, n_features=n_features)
        descripcion = f"n_{n_features}"

    elif metodo == 'lags':
        features_seleccionadas = seleccionar_lags_optimos(df_adv)
        descripcion = "lags_optimos"

    elif metodo == 'pca':
        n_components = kwargs.get('n_components', 10)
        df_pca, pca_cols, pca_model, explained_variance = aplicar_pca(
            df_adv, cols, n_components=n_components
        )
        descripcion = f"pca_{n_components}"
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

        archivo_pca = os.path.join(RUTA_PROCESADOS, f"dengue_pca_{n_components}comp_{timestamp}.xlsx")
        df_pca.to_excel(archivo_pca, index=False)
        print(f"  ✅ Dataset PCA guardado: {archivo_pca}")

        archivo_comp = os.path.join(RUTA_PROCESADOS, f"pca_componentes_{timestamp}.xlsx")
        pd.DataFrame({
            'componente': pca_cols,
            'varianza_explicada': explained_variance,
            'varianza_acumulada': np.cumsum(explained_variance)
        }).to_excel(archivo_comp, index=False)
        print(f"  ✅ Componentes PCA guardados: {archivo_comp}")

        archivo_loadings = os.path.join(RUTA_PROCESADOS, f"pca_loadings_{timestamp}.xlsx")
        df_loadings = pd.DataFrame(
            pca_model.components_.T,
            columns=pca_cols,
            index=cols
        )
        df_loadings.to_excel(archivo_loadings)
        print(f"  ✅ PCA loadings guardados: {archivo_loadings}")

        # Se devuelven también las rutas de archivos para que la tabla
        # comparativa final quede completa (antes se perdían)
        archivos_pca = (archivo_pca, archivo_comp, archivo_loadings)
        return df_pca, pca_cols, archivos_pca

    elif metodo == 'combinado':
        print("\n  Paso 1: Selección por importancia...")
        temp_selected, _ = seleccionar_por_importancia(df_adv, cols, threshold=0.005)

        print("\n  Paso 2: RFE sobre features seleccionadas...")
        n_final = kwargs.get('n_final', 15)
        features_seleccionadas = seleccionar_rfe(df_adv, temp_selected, n_features=n_final)
        descripcion = f"combinado_n{n_final}"

    if len(features_seleccionadas) == 0:
        print("  ⚠️ No se seleccionaron features, usando todas...")
        features_seleccionadas = cols
        descripcion = "todas_las_features"

    print(f"\n2️⃣ Guardando datasets reducidos...")
    archivos = guardar_dataset_reducido(
        df_adv,
        features_seleccionadas,
        metodo,
        total_features_originales=len(cols),
        descripcion=descripcion
    )

    print(f"\n{'='*70}")
    print("RESUMEN DE REDUCCIÓN DIMENSIONAL")
    print(f"{'='*70}")
    print(f"  Método: {metodo}")
    print(f"  Features candidatas: {len(cols)}")
    print(f"  Features seleccionadas: {len(features_seleccionadas)}")
    print(f"  Reducción: {(1 - len(features_seleccionadas)/len(cols))*100:.1f}%")
    print(f"  Archivos guardados en: {RUTA_PROCESADOS}")

    return df_adv, features_seleccionadas, archivos


# ============================================================
# 5. EJECUCIÓN PRINCIPAL
# ============================================================
if __name__ == "__main__":

    print("="*70)
    print("REDUCCIÓN DIMENSIONAL PARA DATOS DE DENGUE")
    print("="*70)

    print("\n Cargando datos (dataset de 172 features con lags ya calculados)...")
    df = cargar_datos()

    resultados = {}

    # Método 1: Importancia (threshold ajustable)
    print("\n" + "="*70)
    print("MÉTODO 1: SELECCIÓN POR IMPORTANCIA")
    print("="*70)
    df_reducido, features_imp, archivos_imp = reducir_dimensionalidad_y_guardar(
        df, metodo='importancia', threshold=0.005
    )
    resultados['importancia'] = {
        'features': features_imp, 'n_features': len(features_imp),
        'archivos': archivos_imp
    }

    # Método 2: RFE
    print("\n" + "="*70)
    print("MÉTODO 2: RECURSIVE FEATURE ELIMINATION (RFE)")
    print("="*70)
    df_reducido, features_rfe, archivos_rfe = reducir_dimensionalidad_y_guardar(
        df, metodo='rfe', n_features=20
    )
    resultados['rfe'] = {
        'features': features_rfe, 'n_features': len(features_rfe),
        'archivos': archivos_rfe
    }

    # Método 3: Lags óptimos
    print("\n" + "="*70)
    print("MÉTODO 3: LAGS ÓPTIMOS")
    print("="*70)
    df_reducido, features_lags, archivos_lags = reducir_dimensionalidad_y_guardar(
        df, metodo='lags'
    )
    resultados['lags'] = {
        'features': features_lags, 'n_features': len(features_lags),
        'archivos': archivos_lags
    }

    # Método 4: PCA
    print("\n" + "="*70)
    print("MÉTODO 4: PCA")
    print("="*70)
    df_pca, componentes_pca, archivos_pca = reducir_dimensionalidad_y_guardar(
        df, metodo='pca', n_components=10
    )
    resultados['pca'] = {
        'features': componentes_pca, 'n_features': len(componentes_pca),
        'archivos': archivos_pca
    }

    # Método 5: Combinado (Importancia + RFE)
    print("\n" + "="*70)
    print("MÉTODO 5: COMBINADO (Importancia + RFE)")
    print("="*70)
    df_reducido, features_comb, archivos_comb = reducir_dimensionalidad_y_guardar(
        df, metodo='combinado', n_final=15
    )
    resultados['combinado'] = {
        'features': features_comb, 'n_features': len(features_comb),
        'archivos': archivos_comb
    }

    # ============================================================
    # 6. RESUMEN FINAL COMPARATIVO
    # ============================================================
    print("\n" + "="*70)
    print("RESUMEN COMPARATIVO DE TODOS LOS MÉTODOS")
    print("="*70)

    # Total de features candidatas (172 originales + derivadas), recalculado
    # una vez para usar como base común de comparación
    df_adv_ref = crear_features_avanzadas(df)
    total_candidatas = len([c for c in df_adv_ref.columns if c not in COLUMNAS_NO_FEATURE])

    tabla_comparativa = pd.DataFrame({
        'Método': list(resultados.keys()),
        'Features seleccionadas': [r['n_features'] for r in resultados.values()],
        'Reducción (%)': [f"{(1 - r['n_features']/total_candidatas)*100:.1f}%"
                          for r in resultados.values()]
    })
    print(tabla_comparativa.to_string(index=False))

    archivo_comparativa = os.path.join(RUTA_PROCESADOS,
        f"comparativa_metodos_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
    tabla_comparativa.to_excel(archivo_comparativa, index=False)
    print(f"\n Tabla comparativa guardada: {archivo_comparativa}")

    archivo_todos_atributos = os.path.join(RUTA_PROCESADOS,
        f"todos_atributos_seleccionados_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")

    with pd.ExcelWriter(archivo_todos_atributos) as writer:
        for metodo, info in resultados.items():
            df_atributos = pd.DataFrame({
                'feature': info['features'],
                'orden': range(1, len(info['features']) + 1)
            })
            df_atributos.to_excel(writer, sheet_name=metodo, index=False)
    print(f"Todos los atributos guardados en: {archivo_todos_atributos}")

    print("\n" + "="*70)
    print("🎉 PROCESO DE REDUCCIÓN DIMENSIONAL COMPLETADO")
    print("="*70)
    print(f" Todos los archivos guardados en: {RUTA_PROCESADOS}")
    print("\nArchivos generados:")
    print("  - dengue_reducido_[metodo]_[timestamp].xlsx (datos reducidos)")
    print("  - dengue_reducido_[metodo]_[timestamp]_atributos.xlsx (lista de atributos)")
    print("  - dengue_reducido_[metodo]_[timestamp]_resumen.xlsx (resumen del método)")
    print("  - comparativa_metodos_[timestamp].xlsx (comparación entre métodos)")
    print("  - todos_atributos_seleccionados_[timestamp].xlsx (todos los atributos)")
    print("  - importancias_features_[timestamp].xlsx (importancias completas)")
    print("  - pca_componentes_[timestamp].xlsx (componentes PCA)")
    print("  - pca_loadings_[timestamp].xlsx (matriz de loadings PCA)")


REDUCCIÓN DIMENSIONAL PARA DATOS DE DENGUE

 Cargando datos (dataset de 172 features con lags ya calculados)...
Datos reales cargados (270 filas, 172 columnas).

MÉTODO 1: SELECCIÓN POR IMPORTANCIA

REDUCCIÓN DIMENSIONAL - MÉTODO: IMPORTANCIA

1️⃣ Creando features avanzadas (agregaciones, ratios, estacionalidad)...
  Features candidatas totales (172 originales + derivadas): 202
  Importancia > 0.005: seleccionadas 3 features
  Top 5: ['casos_media_2w', 'casos_max_2w', 'casos_media_4w']
  ✅ Importancias guardadas: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados\importancias_features_20260902_175155.xlsx

2️⃣ Guardando datasets reducidos...
  ✅ Dataset reducido guardado: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados\dengue_reducido_importancia_threshold_0.005_20260902_175155.xlsx
  ✅ Lista de atributos guardada: C:\Users\marco\Documentos\investigacion\machine_learning_idali

c:\Users\marco\Documentos\investigacion\machine_learning_idalina\.venv\Lib\site-packages\sklearn\feature_selection\_rfe.py:300: UserWarning: Found n_features_to_select=15 > n_features=3. There will be no feature selection and all features will be kept.
  warnings.warn(


     Método  Features seleccionadas Reducción (%)
importancia                       3         98.5%
        rfe                      20         90.1%
       lags                      12         94.1%
        pca                      10         95.0%
  combinado                       3         98.5%

 Tabla comparativa guardada: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados\comparativa_metodos_20260902_175402.xlsx
Todos los atributos guardados en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados\todos_atributos_seleccionados_20260902_175402.xlsx

🎉 PROCESO DE REDUCCIÓN DIMENSIONAL COMPLETADO
 Todos los archivos guardados en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados

Archivos generados:
  - dengue_reducido_[metodo]_[timestamp].xlsx (datos reducidos)
  - dengue_reducido_[metodo]_[timestamp]_atributos.xlsx (lista 